# DLW 数值分析 report

以 DLW 单孤子与二孤子的解析解为参照，比较 SD、SD2 与 FD 三种空间方案、Euler 与 RK4 时间算法，以及固定网格与自适应动网格。误差在共同物理点上评价，双场结果与局部误差曲线采用相同的计算配置。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.interpolate import CubicSpline
from scipy.sparse import bmat, csc_matrix, coo_matrix, diags
from scipy.sparse.linalg import splu
from IPython.display import display

plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.linewidth': .7, 'axes.spines.top': False,
    'axes.spines.right': False, 'xtick.direction': 'in',
    'ytick.direction': 'in', 'figure.dpi': 110})

## 1　模型与实验设计

连续 DLW 方程为

$$\begin{aligned}
u_{yt}&=-\partial_x[(u+2a)u_y]-v_{xx},\\
v_t&=-\partial_x[(u+2a)v-4u]-u_{xxy}.
\end{aligned}$$

采用 Sheng–Yu 原文的三组孤子参数，取 $a=2$、$c_i=1$，初相位为零。A、B 分别对应原文图 1(a)、1(b)，C 对应图 3。计算区间为 $x\in[-20,20)$、$y\in[-1.5,1.5]$；$N_x=256$、$h_y=1/8$，共 24 个 $y$ 中点层。时间步长为 $\Delta t=1.25\times10^{-4}$，终止时刻为 $T=0.01$。

In [ ]:
CASES = {'A': ((1.,), (2.,)), 'B': ((4.,), (-3.,)),
         'C': ((6., 4.), (-5., -3.))}
CONFIG = dict(a=2., nx=256, L=40., h=.125, dt=.000125, T=.01,
              yhalf=1.5, eval_half=10., eval_points=4001)
MODELS = ('SD', 'SD2', 'FD')
COLORS = dict(SD='#1776bc', SD2='#d47b19', FD='#38965f')
pd.DataFrame([{'算例': k, 'p': p, 'q': q} for k, (p, q) in CASES.items()])

### 1.1　孤子参照

记 $S_i=p_i+q_i$、$\omega_i=q_i^2-p_i^2$、$\ell_i=(p_i-a)^{-1}+(q_i+a)^{-1}$，并令 $\theta_i=S_ix+\omega_it+\ell_i y$。单孤子的 $g=1+e^{\theta_1}/S_1$；二孤子的

$$g=1+\frac{e^{\theta_1}}{S_1}+\frac{e^{\theta_2}}{S_2}
+\frac{(p_1-p_2)(q_1-q_2)e^{\theta_1+\theta_2}}
{(p_1+q_1)(p_1+q_2)(p_2+q_1)(p_2+q_2)}.$$

$f$ 在各指数项中加入因子 $\gamma_i=-(p_i-a)/(q_i+a)$。物理场取

$$u_*=2\partial_x(\log f-\log g),\qquad
v_*=2\partial_x\partial_y(\log f+\log g).$$

用对数和计算正系数 tau 函数；对数的一阶、二阶导数分别是指数率的加权均值、协方差。

In [ ]:
class Exact:
    def __init__(self, case, h, a=2.):
        p, q = map(np.asarray, CASES[case])
        self.h = h
        s, omega, ell = p+q, q*q-p*p, 1/(p-a)+1/(q+a)
        gamma = np.log(-(p-a)/(q+a))
        if len(p) == 1:
            self.coeff = np.array([1., 1/s[0]])
            subsets = np.array([[0.], [1.]])
        else:
            cross = ((p[0]-p[1])*(q[0]-q[1]) /
                ((p[0]+q[0])*(p[0]+q[1])*(p[1]+q[0])*(p[1]+q[1])))
            self.coeff = np.array([1., 1/s[0], 1/s[1], cross])
            subsets = np.array([[0., 0.], [1., 0.], [0., 1.], [1., 1.]])
        self.rates = subsets @ np.stack((s, omega, ell), axis=1)
        self.gamma = subsets @ gamma
        self._key, self._cache = None, {}

    def tau(self, js, x, t, is_f=False):
        y = (np.atleast_1d(js)+.5)*self.h
        logs = (np.log(self.coeff)[:, None, None]
            + self.rates[:, 0, None, None]*np.atleast_1d(x)[None, None, :]
            + self.rates[:, 1, None, None]*t
            + self.rates[:, 2, None, None]*y[None, :, None])
        if is_f:
            logs = logs + self.gamma[:, None, None]
        total = logsumexp(logs, axis=0)
        return total, np.exp(logs-total[None, :, :])

    def mean(self, w, k):
        return np.einsum('i,ijk->jk', self.rates[:, k], w)

    def cov(self, w, a, b):
        return (np.einsum('i,ijk->jk', self.rates[:, a]*self.rates[:, b], w)
                - self.mean(w, a)*self.mean(w, b))

    def uv(self, js, x, t):
        key = (float(t), np.asarray(x).tobytes())
        if key != self._key:
            self._key, self._cache = key, {}
        sub = tuple(np.atleast_1d(js))
        if sub not in self._cache:
            _, f = self.tau(js, x, t, True)
            _, g = self.tau(js, x, t)
            self._cache[sub] = (2*(self.mean(f, 0)-self.mean(g, 0)),
                2*(self.cov(f, 0, 2)+self.cov(g, 0, 2)))
        return tuple(v.copy() for v in self._cache[sub])

## 2　空间离散与边界

在 $y$ 方向使用交错差分：$\delta_-f_j=(f_j-f_{j-1})/h$，$\delta_0f_j=(f_{j+1}-f_{j-1})/(2h)$，$M_-f_j=(f_j+f_{j-1})/2$。均匀计算坐标上的四阶中心差分为

$$D_\xi f_i=\frac{f_{i-2}-8f_{i-1}+8f_{i+1}-f_{i+2}}{12\Delta\xi}.$$

节点取 $x_i=\xi_i+s_i$、$J_i=1+D_\xi s_i$，物理导数为 $D_1=J^{-1}D_\xi$、$D_2=D_1(D_1)$。下边界和外层背景取解析值；上侧对数值解与背景之差作二次外推。$x$ 端点按原实现延拓。

In [ ]:
class Grid:
    def __init__(self, n, L):
        self.n, self.L, self.dx = n, L, L/n
        self.xi = -L/2 + np.arange(n)*self.dx
        self.set_s(np.zeros(n))

    def dxi(self, f):
        return (np.roll(f, 2, axis=-1)-8*np.roll(f, 1, axis=-1)
            +8*np.roll(f, -1, axis=-1)-np.roll(f, -2, axis=-1))/(12*self.dx)

    def set_s(self, s):
        self.s = np.asarray(s).copy()
        self.x, self.J = self.xi+self.s, 1+self.dxi(self.s)
        if self.J.min() <= 0 or np.diff(np.r_[self.x, self.x[0]+self.L]).min() <= 0:
            raise ValueError('网格 Jacobian 非正或节点交叉')

    def d1(self, f):
        return self.dxi(f)/self.J

    def d2(self, f):
        return self.d1(self.d1(f))

def delta0(u, ghosts, h):
    ext = np.concatenate((ghosts[0][None, :], u, ghosts[1][None, :]))
    return (ext[2:]-ext[:-2])/(2*h)

def ghost_u(G, js, x, t, u):
    gl, gr = G.uv([js[0]-1, js[-1]+1], x, t)[0]
    # 上侧外推扰动；解析背景与内部数值解分开处理。
    e = u[-3:]-G.uv(js[-3:], x, t)[0]
    return gl, gr+3*e[-1]-3*e[-2]+e[-3]

### 2.1　SD 与 FD

两种方案都以 $P=\delta_-u$ 恢复 $u$。SD 取 $W=v-\delta_0u$，并令

$$H=\frac{u^2}{2}+2au+h^2\left(\frac{W^2}{32}-\frac W4\right).$$

其右端为 $P_t=-\delta_-D_1H-D_2(M_-v-h^2\Delta_hP/4)$、$W_t=-D_1[(u+2a)W-4u]+D_2W$。FD 则直接更新 $P,v$，取 $P_t=-\delta_-D_1(u^2/2+2au)-D_2M_-v$、$v_t=-D_1[(u+2a)v]-D_2\delta_0u+4D_1u$。初始双场来自同一连续解析解。

In [ ]:
class PhysicalModel:
    def __init__(self, case, model, config):
        self.kind, self.h, self.a = model, config['h'], config['a']
        self.X = Grid(config['nx'], config['L'])
        self.G = Exact(case, self.h, self.a)
        n = round(config['yhalf']/self.h)
        self.js = np.arange(-n, n)
        self.y = (self.js+.5)*self.h
        self.shape = (len(self.js), self.X.n)
        self.np = (len(self.js)-1)*self.X.n

    def pack(self, p, w):
        return np.r_[p.ravel(), w.ravel()]

    def unpack(self, z):
        return z[:self.np].reshape(self.shape[0]-1, -1), z[self.np:].reshape(self.shape)

    def recover_u(self, p, t):
        base = self.G.uv([self.js[0]], self.X.x, t)[0][0]
        return np.vstack((base, base+self.h*np.cumsum(p, axis=0)))

    def initial(self):
        u, v = self.G.uv(self.js, self.X.x, 0.)
        ghosts = self.G.uv([self.js[0]-1, self.js[-1]+1], self.X.x, 0.)[0]
        w = v-delta0(u, ghosts, self.h) if self.kind == 'SD' else v
        return self.pack(np.diff(u, axis=0)/self.h, w)

    def fields(self, z, t):
        p, w = self.unpack(z)
        u = self.recover_u(p, t)
        ghosts = ghost_u(self.G, self.js, self.X.x, t, u)
        return u, w+delta0(u, ghosts, self.h) if self.kind == 'SD' else w

    def rhs(self, t, z):
        p, w = self.unpack(z)
        u = self.recover_u(p, t)
        gl, gr = ghost_u(self.G, self.js, self.X.x, t, u)
        uy = delta0(u, (gl, gr), self.h)
        d1, d2, h, a = self.X.d1, self.X.d2, self.h, self.a
        if self.kind == 'SD':
            v = w+uy
            H = u*u/2+2*a*u+h*h*(w*w/32-w/4)
            pe = np.vstack(((u[0]-gl)/h, p, (gr-u[-1])/h))
            lap = (pe[2:]-2*pe[1:-1]+pe[:-2])/(h*h)
            fp = -np.diff(d1(H), axis=0)/h-d2((v[1:]+v[:-1])/2-h*h*lap/4)
            fw = -d1((u+2*a)*w-4*u)+d2(w)
        else:
            fp = -np.diff(d1(u*u/2+2*a*u), axis=0)/h-d2((w[1:]+w[:-1])/2)
            fw = -d1((u+2*a)*w)-d2(uy)+4*d1(u)
        return self.pack(fp, fw)

### 2.2　SD2 的物理初值与下边界

SD2 以 $Q,R$ 为演化变量。为使三个方案的物理初值一致，离散求解

$$D_1Q_j^0=\frac12u_j^0Q_j^0,\qquad Q_{j,0}^0=1,\qquad
R_j^0=\frac{1-(v_j^0-\delta_0u_j^0)/4}{Q_j^0}.$$

跨 $x$ 端点的跃变量和 $Q$ 同时求出。演化时只更新下边界的提升，内部 $Q,R$ 按演化方程推进；下边界的时间导数由离散提升式求导得到。

In [ ]:
class SD2Lift(PhysicalModel):
    def __init__(self, case, config):
        super().__init__(case, 'SD2', config)
        n, dx = self.X.n, self.X.dx
        ii = np.arange(n)
        self.D = coo_matrix((np.concatenate([np.full(n, c/(12*dx))
            for c in (1, -8, 8, -1)]), (np.tile(ii, 4),
            np.concatenate([(ii+k) % n for k in (-2, -1, 1, 2)]))),
            shape=(n, n)).tocsc()
        self.b = np.zeros(n)
        self.b[:2], self.b[-2:] = [7, -1], [-1, 7]
        self.b /= 12*dx
        self.norm = csc_matrix(([1.], ([0], [0])), shape=(1, n))
        self.initial_right, self._boundary_key = None, None
        self.jump = self.rjump = 0.

    def dx(self, f, jump=0.):
        # 跨 x 端点时使用 f(x+L)=f(x)+jump。
        def shift(k):
            a = np.roll(f, k, axis=-1).copy()
            if k > 0:
                a[..., :k] -= jump
            else:
                a[..., k:] += jump
            return a
        return (shift(2)-8*shift(1)+8*shift(-1)-shift(-2))/(12*self.X.dx*self.X.J)

    def lift(self, u):
        A = bmat([[self.D-diags(.5*self.X.J*u), csc_matrix(self.b[:, None])],
                  [self.norm, csc_matrix((1, 1))]], format='csc')
        lu = splu(A)
        sol = lu.solve(np.r_[np.zeros(self.X.n), 1.])
        q, jump = sol[:-1], sol[-1]
        if q.min() <= 0 or 1+jump <= 0:
            raise ValueError('SD2 提升后的 Q 或端点比值非正')
        return q, jump, lu

    def boundary(self, t):
        key = (float(t), self.X.x.tobytes(), self.X.J.tobytes())
        if key != self._boundary_key:
            _, f = self.G.tau([self.js[0]], self.X.x, t, True)
            _, g = self.G.tau([self.js[0]], self.X.x, t)
            u = 2*(self.G.mean(f, 0)-self.G.mean(g, 0))[0]
            ut = 2*(self.G.cov(f, 0, 1)-self.G.cov(g, 0, 1))[0]
            ux = 2*(self.G.cov(f, 0, 0)-self.G.cov(g, 0, 0))[0]
            q, jump, lu = self.lift(u)
            self._boundary_key = key
            self._boundary_value = q, jump, lu, u, ut, ux
        q, jump, lu, u, ut, ux = self._boundary_value
        if self.initial_right is not None:
            right = self.initial_right*(1+jump)/self.initial_right[0]
            self.jump, self.rjump = (right-1)[:, None], (1/right-1)[:, None]
        return self._boundary_value

    def boundary_derivative(self, t, velocity):
        q, jump, lu, u, ut, ux = self.boundary(t)
        rhs = .5*(self.D@velocity*u+self.X.J*(ut+velocity*ux))*q
        material_qt = lu.solve(np.r_[rhs, 0.])[:-1]
        qx = (self.D@q+self.b*jump)/self.X.J
        return material_qt-velocity*qx

    def pack(self, q, r):
        return np.r_[q[1:].ravel(), r.ravel()]

    def unpack(self, z, t):
        q = np.vstack((self.boundary(t)[0], z[:self.np].reshape(self.shape[0]-1, -1)))
        return q, z[self.np:].reshape(self.shape)

    def initial(self):
        u, v = self.G.uv(self.js, self.X.x, 0.)
        lifted = [self.lift(row) for row in u]
        q = np.array([v[0] for v in lifted])
        self.initial_right = 1+np.array([v[1] for v in lifted])
        self._boundary_key = None
        self.boundary(0.)
        ghosts = self.G.uv([self.js[0]-1, self.js[-1]+1], self.X.x, 0.)[0]
        r = (1-(v-delta0(u, ghosts, self.h))/4)/q
        return self.pack(q, r)

### 2.3　SD2 演化

令 $S=QR$、$G=h^2(S^2-1)/4$、$m_j=M_{j,x}$。由下边界确定 $m_0$，再使用 $m_{j+1}=m_j-hD_1S_j$，令 $A_j=m_j+m_{j+1}$。演化与物理场恢复为

$$\begin{aligned}
Q_t&=-D_2Q-2aD_1Q-(A+G)Q,\\
R_t&= D_2R-2aD_1R+(A+G)R,\\
u&=2(D_1Q)/Q,\qquad v=4(1-QR)+\delta_0u.
\end{aligned}$$

In [ ]:
class SD2Model(SD2Lift):
    def fields(self, z, t):
        q, r = self.unpack(z, t)
        u = 2*self.dx(q, self.jump)/q
        ghosts = ghost_u(self.G, self.js, self.X.x, t, u)
        return u, 4*(1-q*r)+delta0(u, ghosts, self.h)

    def stage(self, t, state, mesh):
        self.X.set_s(state[-self.X.n:])
        q, r = self.unpack(state[:-self.X.n], t)
        if q.min() <= 0:
            raise ValueError('演化后的 Q 非正')
        qx, rx = self.dx(q, self.jump), self.dx(r, self.rjump)
        w = q*r
        velocity = np.zeros(self.X.n)
        if mesh == 'moving':
            density = w.mean(axis=0)
            flux = ((2*qx/q+2*self.a)*w-self.dx(w)-2*self.a).mean(axis=0)
            if density.min() <= 0:
                raise ValueError('动网格监测密度非正')
            velocity = (flux-flux[0])/density
        qt0 = self.boundary_derivative(t, velocity)
        qxx, rxx, wx = self.dx(qx), self.dx(rx), self.dx(w)
        H = self.h**2/4*(w*w-1)
        A0 = -(qt0+qxx[0]+2*self.a*qx[0])/q[0]-H[0]
        mx0 = (A0+self.h*wx[0])/2
        mx = mx0-self.h*np.vstack((np.zeros(self.X.n), np.cumsum(wx, axis=0)))
        A = mx[:-1]+mx[1:]
        qt = -qxx-2*self.a*qx-(A+H)*q
        rt = rxx-2*self.a*rx+(A+H)*r
        return np.r_[self.pack(qt+velocity*qx, rt+velocity*rx), velocity]

### 2.4　网格策略

初始动网格按密度 $\rho_j=1-(v_j-\delta_0u_j)/4$ 的层平均累积积分等分。记

$$\bar\rho=\frac1{N_y}\sum_j\rho_j,\qquad
\bar q=\frac1{N_y}\sum_j[(u_j+2a)\rho_j-D_1\rho_j-2a],$$

取 $V_i=(\bar q_i-\bar q_0)/\bar\rho_i$。节点满足 $\dot x=V$，场状态满足 $\dot z=\mathcal F(t,z)+VD_1z$。固定网格取 $V=0$。

In [ ]:
class Problem:
    def __init__(self, case, model, mesh, config=CONFIG):
        self.mesh, self.model = mesh, model
        self.m = SD2Model(case, config) if model == 'SD2' else PhysicalModel(case, model, config)
        self.X = self.m.X

    def initial(self):
        if self.mesh == 'moving':
            x = np.linspace(-self.X.L/2, self.X.L/2, 40001)
            u, v = self.m.G.uv(self.m.js, x, 0.)
            gh = self.m.G.uv([self.m.js[0]-1, self.m.js[-1]+1], x, 0.)[0]
            density = 1-(v-delta0(u, gh, self.m.h)).mean(axis=0)/4
            if density.min() <= 0:
                raise ValueError('初始监测密度非正')
            mass = np.r_[0, np.cumsum((density[:-1]+density[1:])*np.diff(x)/2)]
            nodes = np.interp(np.arange(self.X.n)/self.X.n*mass[-1], mass, x)
            self.X.set_s(nodes-self.X.xi)
        return np.r_[self.m.initial(), self.X.s]

    def fields(self, state, t):
        self.X.set_s(state[-self.X.n:])
        return self.m.fields(state[:-self.X.n], t)

    def stage(self, t, state):
        if self.model == 'SD2':
            return self.m.stage(t, state, self.mesh)
        z = state[:-self.X.n]
        self.X.set_s(state[-self.X.n:])
        physical = self.m.rhs(t, z)
        velocity = np.zeros(self.X.n)
        if self.mesh == 'moving':
            u, v = self.m.fields(z, t)
            ghosts = ghost_u(self.m.G, self.m.js, self.X.x, t, u)
            rho = 1-(v-delta0(u, ghosts, self.m.h))/4
            density = rho.mean(axis=0)
            flux = ((u+2*self.m.a)*rho-self.X.d1(rho)-2*self.m.a).mean(axis=0)
            if density.min() <= 0:
                raise ValueError('动网格监测密度非正')
            velocity = (flux-flux[0])/density
            p, w = self.m.unpack(z)
            physical += self.m.pack(self.X.d1(p)*velocity, self.X.d1(w)*velocity)
        return np.r_[physical, velocity]

## 3　时间推进与误差评价

Euler 采用 $z^{n+1}=z^n+\Delta t\,\mathcal F(t_n,z^n)$；RK4 的四个时间级均恢复当前网格、场和下边界，状态与节点同步更新。

在 $x\in[-10,10]$ 的 4001 个等距点及全部 $y$ 层上，三次样条重构数值场，计算

$$E_f(T)=\max_{(x,y)\in\mathcal G}|f_h(x,y,T)-f_*(x,y,T)|,\qquad f\in\{u,v\}.$$

这些是指定评价网格上的最大误差。节点次序、Jacobian 与监测密度须保持正性；未到达共同终点的轨道不进入误差排名。

In [ ]:
def step(fun, t, state, dt, method):
    if method == 'Euler':
        return state+dt*fun(t, state)
    k1 = fun(t, state)
    k2 = fun(t+dt/2, state+dt*k1/2)
    k3 = fun(t+dt/2, state+dt*k2/2)
    k4 = fun(t+dt, state+dt*k3)
    return state+dt*(k1+2*k2+2*k3+k4)/6

def solve(case, model, method, mesh, config=CONFIG):
    p = Problem(case, model, mesh, config)
    state = p.initial()
    initial = p.fields(state, 0.)
    initial_ref = p.m.G.uv(p.m.js, p.X.x, 0.)
    initial_error = max(float(np.max(abs(u-v))) for u, v in zip(initial, initial_ref))
    nsteps = round(config['T']/config['dt'])
    if not np.isclose(nsteps*config['dt'], config['T'], rtol=0, atol=1e-13):
        raise ValueError('T 必须是 dt 的整数倍')
    reached, reason = 0., ''
    for n in range(nsteps):
        try:
            with np.errstate(over='raise', invalid='raise', divide='raise'):
                candidate = step(p.stage, n*config['dt'], state, config['dt'], method)
                if not np.all(np.isfinite(candidate)) or abs(candidate).max() > 1000:
                    raise ValueError('状态非有限或绝对值超过 1000')
                p.X.set_s(candidate[-p.X.n:])
        except (ValueError, FloatingPointError, OverflowError) as e:
            reason = str(e)
            break
        state, reached = candidate, (n+1)*config['dt']
    xx = np.linspace(-config['eval_half'], config['eval_half'], config['eval_points'])
    native = p.fields(state, reached)
    sampled = [CubicSpline(p.X.x, f, axis=-1)(xx) for f in native]
    exact = p.m.G.uv(p.m.js, xx, reached)
    errors = {f: abs(u-v) for f, u, v in zip(('u', 'v'), sampled, exact)}
    return dict(case=case, model=model, method=method, mesh=mesh,
        reached=reached, completed=np.isclose(reached, config['T']), reason=reason,
        initial_error=initial_error, x=xx, y=p.m.y, fields=sampled, exact=exact,
        errors=errors, max_errors={f: float(e.max()) for f, e in errors.items()},
        min_J=float(p.X.J.min()), native_x=p.X.x.copy(), native_fields=native)

results = {}
def calculate(method, mesh):
    for case in CASES:
        for model in MODELS:
            key = case, model, method, mesh
            results[key] = solve(case, model, method, mesh)
    failed = [r for r in results.values() if not r['completed']]
    if failed:
        display(pd.DataFrame([{k: r[k] for k in ('case', 'model', 'method', 'mesh', 'reached', 'reason')}
                             for r in failed]))
        raise RuntimeError('未到达共同终点，停止配对比较')

def error_table(method, mesh):
    return pd.DataFrame([{'算例': case, '场': field,
        **{m: results[case, m, method, mesh]['max_errors'][field] for m in MODELS}}
        for case in CASES for field in ('u', 'v')]).set_index(['算例', '场'])

## 4　空间方案的比较

固定算例、RK4、均匀网格和时间步长，比较 SD、SD2 与 FD。每行加粗值为较小误差。

In [ ]:
calculate('RK4', 'fixed')
space_table = error_table('RK4', 'fixed')
space_table.style.format('{:.3e}').highlight_min(axis=1, props='font-weight: bold')

## 5　时间算法的比较

在同一空间方案和固定网格下并列 Euler 与 RK4。总误差同时包含空间离散、边界与重构误差，较高的时间阶不保证每个场的总误差都更小。

In [ ]:
calculate('Euler', 'fixed')
time_table = pd.DataFrame([{'算例': case, '方案': model, '场': field,
    **{method: results[case, model, method, 'fixed']['max_errors'][field]
       for method in ('Euler', 'RK4')}}
    for case in CASES for model in MODELS for field in ('u', 'v')]).set_index(['算例', '方案', '场'])
time_table.style.format('{:.3e}').highlight_min(axis=1, props='font-weight: bold')

取 $\Delta t$、$\Delta t/2$、$\Delta t/4$，以相邻时间步长的数值场差估计时间阶：

$$p_f=\log_2\frac{\|f_{\Delta t}-f_{\Delta t/2}\|_{\infty,\mathcal G}}
{\|f_{\Delta t/2}-f_{\Delta t/4}\|_{\infty,\mathcal G}}.$$

下面对三组算例的 SD2 固定网格作 Euler 自收敛比较。

In [ ]:
order_rows = []
for case in CASES:
    fields = [results[case, 'SD2', 'Euler', 'fixed']['fields']]
    for divisor in (2, 4):
        refined = solve(case, 'SD2', 'Euler', 'fixed',
                        dict(CONFIG, dt=CONFIG['dt']/divisor))
        if not refined['completed']:
            raise RuntimeError(refined['reason'])
        fields.append(refined['fields'])
    for i, field in enumerate(('u', 'v')):
        d1 = np.max(abs(fields[0][i]-fields[1][i]))
        d2 = np.max(abs(fields[1][i]-fields[2][i]))
        order_rows.append({'算例': case, '方案': 'SD2', '场': field,
                          'dt 与 dt/2 场差': d1, 'dt/2 与 dt/4 场差': d2,
                          '观测阶': np.log2(d1/d2)})
pd.DataFrame(order_rows).set_index(['算例', '方案', '场']).style.format({
    'dt 与 dt/2 场差': '{:.3e}', 'dt/2 与 dt/4 场差': '{:.3e}', '观测阶': '{:.4f}'})

## 6　固定网格与动网格的比较

保持节点数、时间步长、RK4 与空间方案一致，比较均匀固定网格与自适应动网格。各场在同一组物理点上评价。

In [ ]:
calculate('RK4', 'moving')
mesh_table = pd.DataFrame([{'算例': case, '方案': model, '场': field,
    **{mesh: results[case, model, 'RK4', mesh]['max_errors'][field]
       for mesh in ('fixed', 'moving')}}
    for case in CASES for model in MODELS for field in ('u', 'v')]).set_index(['算例', '方案', '场'])
mesh_table.style.format('{:.3e}').highlight_min(axis=1, props='font-weight: bold')

## 7　Euler 局部误差曲线

取 $x\in[-1,1]$，画出 $e_f(x)=\max_j|f_h(x,y_j,T)-f_*(x,y_j,T)|$。蓝、橙、绿分别对应 SD、SD2、FD；每组算例的同一场在固定网格与动网格图中使用相同纵轴尺度。

In [ ]:
calculate('Euler', 'moving')
for case in CASES:
    fig, axes = plt.subplots(2, 2, figsize=(10.5, 7), layout='constrained')
    for i, field in enumerate(('u', 'v')):
        upper = max(results[case, model, 'Euler', mesh]['errors'][field][:,
            (results[case, model, 'Euler', mesh]['x'] >= -1) &
            (results[case, model, 'Euler', mesh]['x'] <= 1)].max()
            for model in MODELS for mesh in ('fixed', 'moving'))
        for j, mesh in enumerate(('fixed', 'moving')):
            ax = axes[i, j]
            for model in MODELS:
                r = results[case, model, 'Euler', mesh]
                take = (r['x'] >= -1) & (r['x'] <= 1)
                curve = r['errors'][field].max(axis=0)
                ax.plot(r['x'][take], curve[take], color=COLORS[model], lw=1.3, label=model)
            ax.set(xlabel='$x$', ylabel=f'$e_{field}(x)$', xlim=(-1, 1),
                   ylim=(0, upper*1.06), title=f'{field}: {mesh}')
            ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
            ax.grid(alpha=.16, lw=.5)
            ax.legend(frameon=False)
    fig.suptitle(f'Case {case} | Euler | T={CONFIG["T"]:g}')
    plt.show()

## 8　结果与适用范围

配对表给出空间方案的最小误差与动网格相对固定网格的误差比；比值小于 1 表示该场误差降低。

In [ ]:
for case in CASES:
    best_u = space_table.loc[(case, 'u')].idxmin()
    best_v = space_table.loc[(case, 'v')].idxmin()
    print(f'算例 {case}：RK4 固定网格下，u / v 的最小误差方案为 {best_u} / {best_v}。')
mesh_ratios = mesh_table['moving']/mesh_table['fixed']
ratio_table = mesh_ratios.unstack('场').rename(columns={'u': 'u 动格/固定', 'v': 'v 动格/固定'})
display(ratio_table.style.format('{:.3f}'))
print(f'全部 {len(results)} 组主试验到达 T={CONFIG["T"]:g}；'
      f'初始物理场的最大节点差为 {max(r["initial_error"] for r in results.values()):.3e}。')

单孤子 A 的 SD2 固定网格结果对空间分辨率敏感。上述排序限于 $T=0.01$、当前格距、边界闭合及连续解析参照；时间自收敛不能代替空间收敛检验。动网格收益须分别考察 $u,v$，不能由某一场的改善推断双场同时改善。

参考资料：H.-H. Sheng and G.-F. Yu, *Solitons, breathers and rational solutions for a (2+1)-dimensional dispersive long wave system*, Physica D 432 (2022), 133140。算法与参数对应本地原报告 `index.html`，数值实现沿用其统一物理初值、下边界提升及背景相对外推闭合。